# Sweep 11c — Labs Thesis: Lab Configuration Study

**~57 runs ≈ 8–9 hours**

| Group | What | Configs | Runs |
|-------|------|---------|------|
| C | Encoder type: flat vs per_lab_attn | 2 | 6 |
| D | Feature richness: values vs presence-only | 2 | 6 |
| E | Historical attention: on vs off | 2 | 6 |
| F | Drug text embedding + labs | 1 | 3 |
| H | Remaining encoders: per_lab_attn_delta, clinical_bin, isab, lab_as_text | 4 | 12 |
| I | Trajectory encoder: traj_lstm (needs --use_lab_trajectory) | 1 | 3 |
| G | Best combo (fill in after seeing H+I results) | 1 | 3 |

All configs: naked + 200 labs, hgt_ehr_feat, transformer, no copy, no notes.

**Note on Group H encoders:**
- `per_lab_attn_delta`: same as per_lab_attn but with delta (change from last visit) added
- `clinical_bin`: discretizes z-scores into 4 bins (missing/low/normal/high), no raw values
- `isab`: Set Transformer — treats labs as a set, order-invariant attention
- `lab_as_text`: uses precomputed per-lab text embeddings binned by value; needs `lab_text_embeddings.pt`

**Note on Group I:** `traj_lstm` processes each lab as a longitudinal sequence across visits.
Requires `--use_lab_trajectory` to populate trajectory tensors in the dataset.

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn

In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            # per_lab_attn needs description embeddings
            lab_emb = "./data/processed/labs/top_200/lab_description_embeddings.npy"
            print(f"  per_lab_attn embeddings: {'OK' if os.path.exists(lab_emb) else 'MISSING'}")
            # lab_as_text needs lab_text_embeddings.pt at data/processed/ root
            text_emb_src = "./data/processed/labs/top_200/lab_text_embeddings.pt"
            text_emb_dst = "./data/processed/lab_text_embeddings.pt"
            if os.path.exists(text_emb_src) and not os.path.exists(text_emb_dst):
                os.symlink(os.path.abspath(text_emb_src), text_emb_dst)
                print(f"  lab_text_embeddings.pt linked for lab_as_text encoder")
            elif os.path.exists(text_emb_dst):
                print(f"  lab_text_embeddings.pt: OK")
            else:
                print(f"  WARNING: lab_text_embeddings.pt missing — lab_as_text will use random init")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

print("Setup done:", os.getcwd())

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

BACKBONE_MODEL = {
    "graph_encoder_type": "hgt_ehr_feat",
    "hgt_layers": 1,
    "pos_weight_cap": 15.0,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
)
LAB200_FLAGS = "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"
BASE_MODEL_OV = {"copy_mechanism": False, **BACKBONE_MODEL}
BASE_PREP_OV  = {"note_method": "none", "lab_dim": 400}

SEEDS = [42, 123, 456]
reports_dir = Path("experiment_reports/sweep_11c_lab_config")
results_log = []

print("Config helpers ready.")

In [ ]:
import subprocess
from pathlib import Path

Path("smoke_s11c").mkdir(exist_ok=True)

SMOKE = [
    {"name": "C_per_lab",          "flags": f"{LAB200_FLAGS} --lab_encoder_type per_lab_attn"},
    {"name": "D_presence",         "flags": f"{LAB200_FLAGS} --lab_values_zeroed"},
    {"name": "E_no_hist",          "flags": f"{LAB200_FLAGS} --no_historical_attention"},
    {"name": "F_drug_text",        "flags": f"{LAB200_FLAGS} --drug_text_pickle data/processed/drug_text_embeddings.pkl"},
    {"name": "H_attn_delta",       "flags": f"{LAB200_FLAGS} --lab_encoder_type per_lab_attn_delta"},
    {"name": "H_clinical_bin",     "flags": f"{LAB200_FLAGS} --lab_encoder_type clinical_bin"},
    {"name": "H_isab",             "flags": f"{LAB200_FLAGS} --lab_encoder_type isab"},
    {"name": "H_lab_as_text",      "flags": f"{LAB200_FLAGS} --lab_encoder_type lab_as_text"},
    {"name": "I_traj_lstm",        "flags": f"{LAB200_FLAGS} --lab_encoder_type traj_lstm --use_lab_trajectory"},
]

smoke_results = []
for t in SMOKE:
    cfg_path = f"s11c_smoke_{t['name']}.yaml"
    make_config(cfg_path, model_overrides=BASE_MODEL_OV,
                preprocessing_overrides=BASE_PREP_OV, smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {t['flags']} "
           f"--seed 42 --results_dir smoke_s11c/{t['name']}")
    print(f"SMOKE {t['name']}: {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (proc.stdout + proc.stderr).strip().split("\n")[-5:]
    for l in tail: print(l)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {t['name']}")
    print(f">>> {status}\n")

for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("All smoke tests passed.")

## Group C — Lab encoder type (flat vs per_lab_attn)

In [ ]:
import subprocess, gc, torch

def run_group(configs, cfg_yaml_path=None):
    for cfg in configs:
        print(f"\n{'#'*70}\n# {cfg['name']} — {cfg['note']}\n{'#'*70}")
        path = cfg_yaml_path or f"s11c_{cfg['name']}.yaml"
        if cfg_yaml_path is None:
            make_config(path, model_overrides=BASE_MODEL_OV, preprocessing_overrides=BASE_PREP_OV)
        for seed in SEEDS:
            run_name = f"{cfg['name']}_seed{seed}"
            run_dir  = reports_dir / run_name
            run_dir.mkdir(parents=True, exist_ok=True)
            cmd = (f"python -u src/train.py --config {path} "
                   f"{BACKBONE_FLAGS} {cfg['flags']} "
                   f"--seed {seed} --results_dir {run_dir}")
            print(f"\n=== {run_name} ===\n>> {cmd}\n")
            try:
                with open(run_dir / "training_log.txt", "w") as lf:
                    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                            stderr=subprocess.STDOUT, text=True)
                    for line in proc.stdout:
                        print(line, end="")
                        lf.write(line)
                    proc.wait()
                status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
                results_log.append(f"{status}: {run_name}")
            except Exception as e:
                results_log.append(f"CRASH: {run_name}: {e}")
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

# Shared config file for groups that only differ in CLI flags
BASE_CFG = "s11c_base.yaml"
make_config(BASE_CFG, model_overrides=BASE_MODEL_OV, preprocessing_overrides=BASE_PREP_OV)

run_group([
    {"name": "C_flat",         "flags": f"{LAB200_FLAGS} --lab_encoder_type flat",
     "note": "flat MLP encoder (control)"},
    {"name": "C_per_lab_attn", "flags": f"{LAB200_FLAGS} --lab_encoder_type per_lab_attn",
     "note": "per-lab semantic attention (uses labs/top_200/ description embeddings)"},
], BASE_CFG)
print("Group C done.")

## Group D — Lab feature richness (values vs presence-only)

In [ ]:
run_group([
    {"name": "D_full_values",   "flags": LAB200_FLAGS,
     "note": "400d z-scores + presence flags (control)"},
    {"name": "D_presence_only", "flags": f"{LAB200_FLAGS} --lab_values_zeroed",
     "note": "presence-only: z-score columns zeroed"},
], BASE_CFG)
print("Group D done.")

## Group E — Historical attention

In [ ]:
run_group([
    {"name": "E_hist_on",  "flags": LAB200_FLAGS,
     "note": "historical attention ON (default)"},
    {"name": "E_hist_off", "flags": f"{LAB200_FLAGS} --no_historical_attention",
     "note": "historical attention OFF"},
], BASE_CFG)
print("Group E done.")

## Group F — Drug text embedding

In [ ]:
run_group([
    {"name": "F_drug_text", "flags": f"{LAB200_FLAGS} --drug_text_pickle data/processed/drug_text_embeddings.pkl",
     "note": "labs + drug text embeddings (semantic graph edges)"},
], BASE_CFG)
print("Group F done.")

## Group H — Remaining lab encoders

| Encoder | What it does |
|---------|-------------|
| `per_lab_attn_delta` | Same as per_lab_attn + adds Δ (change from last visit) to each lab |
| `clinical_bin` | Discretizes z-scores into 4 bins (missing/low/normal/high) via learned embeddings |
| `isab` | Set Transformer — order-invariant attention over the set of labs |
| `lab_as_text` | Looks up precomputed text embeddings per (lab × bin) — needs `lab_text_embeddings.pt` |

In [ ]:
run_group([
    {"name": "H_per_lab_attn_delta", "flags": f"{LAB200_FLAGS} --lab_encoder_type per_lab_attn_delta",
     "note": "per-lab attention + delta from last visit"},
    {"name": "H_clinical_bin",       "flags": f"{LAB200_FLAGS} --lab_encoder_type clinical_bin",
     "note": "discretized bins (missing/low/normal/high) via learned embeddings"},
    {"name": "H_isab",               "flags": f"{LAB200_FLAGS} --lab_encoder_type isab",
     "note": "Set Transformer ISAB — order-invariant lab attention"},
    {"name": "H_lab_as_text",        "flags": f"{LAB200_FLAGS} --lab_encoder_type lab_as_text",
     "note": "text embeddings per (lab x bin) — uses lab_text_embeddings.pt"},
], BASE_CFG)
print("Group H done.")

## Group I — Trajectory LSTM

`traj_lstm` processes each lab as a longitudinal time-series across patient visits using a shared LSTM.  
Requires `--use_lab_trajectory` to populate trajectory tensors in the dataset loader.  
This is the only encoder that explicitly models how each lab VALUE changes over time.

In [ ]:
# traj_lstm needs its own config because it requires --use_lab_trajectory
run_group([
    {"name": "I_traj_lstm", "flags": f"{LAB200_FLAGS} --lab_encoder_type traj_lstm --use_lab_trajectory",
     "note": "per-lab LSTM over visit trajectory — only encoder modeling lab change over time"},
], BASE_CFG)
print(f"\nAll groups done — {len(results_log)} runs | {sum(1 for r in results_log if 'SUCCESS' in r)} succeeded")

## Group G — Best combo

After reviewing all group results above, set the winner and run 3 seeds.

In [ ]:
# ── EDIT THESE based on C/D/E/F/H/I results ──────────────────────────────────
BEST_ENCODER    = "flat"       # best from C+H+I
BEST_N_LABS     = 200          # best from 11b
BEST_PKL_SUFFIX = "200labs"
USE_PRESENCE_ONLY  = False     # True if D showed Δ_values < 0.002
USE_HIST_ATTN      = True      # False if E showed no benefit
USE_DRUG_TEXT      = False     # True if F showed Δ > 0.002
USE_LAB_TRAJECTORY = False     # True if I_traj_lstm won
# ─────────────────────────────────────────────────────────────────────────────

best_flags = (
    f"--lab_pkl data/processed/lab_vectors_{BEST_PKL_SUFFIX}.pkl "
    f"--num_labs {BEST_N_LABS} --lab_encoder_type {BEST_ENCODER}"
)
if USE_PRESENCE_ONLY:  best_flags += " --lab_values_zeroed"
if not USE_HIST_ATTN:  best_flags += " --no_historical_attention"
if USE_DRUG_TEXT:      best_flags += " --drug_text_pickle data/processed/drug_text_embeddings.pkl"
if USE_LAB_TRAJECTORY: best_flags += " --use_lab_trajectory"

print(f"Best combo flags: {best_flags}")

import subprocess, gc, torch
cfg_path = "s11c_G_best_combo.yaml"
make_config(cfg_path,
            model_overrides={"copy_mechanism": False, **BACKBONE_MODEL},
            preprocessing_overrides={"note_method": "none", "lab_dim": BEST_N_LABS * 2})

for seed in SEEDS:
    run_name = f"G_best_combo_seed{seed}"
    run_dir  = reports_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {best_flags} --seed {seed} --results_dir {run_dir}")
    print(f"\n=== {run_name} ===\n>> {cmd}\n")
    try:
        with open(run_dir / "training_log.txt", "w") as lf:
            proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(line, end="")
                lf.write(line)
            proc.wait()
        status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
        results_log.append(f"{status}: {run_name}")
    except Exception as e:
        results_log.append(f"CRASH: {run_name}: {e}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Group G done.")

In [ ]:
import json, numpy as np
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_11c_lab_config")

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

results = {}
for jp in sorted(reports_dir.rglob("result_*.json")):
    with open(jp) as f: d = json.load(f)
    rn = jp.parent.name; idx = rn.rfind("_seed")
    if idx == -1: continue
    results.setdefault(rn[:idx], []).append(d)

JAC_A0_NAKED = 0.0  # <-- from 11a
JAC_A1_200   = 0.0  # <-- from 11a

def summarize(name):
    runs = results.get(name, [])
    if not runs: return None
    jac = [get_metric(d, "jaccard") for d in runs]
    return {"jac_mean": np.mean(jac), "jac_std": np.std(jac, ddof=1) if len(jac)>1 else 0.0, "n": len(jac)}

def row(name, label, ref):
    s = summarize(name)
    if not s: print(f"  {label:<38} (missing)"); return 0.0
    print(f"  {label:<38} {s['jac_mean']:.4f}±{s['jac_std']:.4f}  Δ={s['jac_mean']-ref:+.4f}  n={s['n']}")
    return s["jac_mean"]

REF = JAC_A1_200
print(f"Reference A1 (200labs flat): {REF:.4f}  |  A0 naked: {JAC_A0_NAKED:.4f}")

print("\n── C: flat vs per_lab_attn ──────────────────────────────────")
jac_c_flat = row("C_flat",          "flat MLP",            REF)
jac_c_attn = row("C_per_lab_attn",  "per_lab_attn",        REF)

print("\n── D: Feature richness ──────────────────────────────────────")
jac_d_full = row("D_full_values",   "full 400d",           REF)
jac_d_pres = row("D_presence_only", "presence only",       REF)

print("\n── E: Historical attention ──────────────────────────────────")
jac_e_on  = row("E_hist_on",  "hist_attn ON",  REF)
jac_e_off = row("E_hist_off", "hist_attn OFF", REF)

print("\n── F: Drug text ─────────────────────────────────────────────")
jac_f = row("F_drug_text", "labs + drug text", REF)

print("\n── H: Remaining encoders ────────────────────────────────────")
jac_h_delta  = row("H_per_lab_attn_delta", "per_lab_attn_delta",   REF)
jac_h_bin    = row("H_clinical_bin",        "clinical_bin",         REF)
jac_h_isab   = row("H_isab",               "isab (Set Transformer)", REF)
jac_h_text   = row("H_lab_as_text",        "lab_as_text",           REF)

print("\n── I: Trajectory LSTM ───────────────────────────────────────")
jac_i = row("I_traj_lstm", "traj_lstm", REF)

print("\n── G: Best combo ────────────────────────────────────────────")
jac_g = row("G_best_combo", "best combo", JAC_A0_NAKED)

# Encoder ranking
encoder_scores = {
    "flat": jac_c_flat, "per_lab_attn": jac_c_attn,
    "per_lab_attn_delta": jac_h_delta, "clinical_bin": jac_h_bin,
    "isab": jac_h_isab, "lab_as_text": jac_h_text, "traj_lstm": jac_i,
}
best_enc = max(encoder_scores, key=encoder_scores.get)
print(f"\n{'='*75}")
print("LAB ENCODER RANKING")
print(f"{'='*75}")
for enc, jac in sorted(encoder_scores.items(), key=lambda x: -x[1]):
    marker = " ← WINNER" if enc == best_enc else ""
    print(f"  {enc:<25} {jac:.4f}  Δ={jac-REF:+.4f}{marker}")

In [ ]:
import zipfile
from pathlib import Path

zip_name = "reports_sweep_11c_lab_config.zip"
rd = Path("experiment_reports/sweep_11c_lab_config")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rd.rglob("result_*.json")): zf.write(p, p.relative_to(rd))
    for p in sorted(rd.rglob("training_log.txt")): zf.write(p, p.relative_to(rd))
print(f"Exported → {zip_name}")